In [1]:
import argparse
import os
import pathlib
import sys
import uuid

import duckdb
import pandas as pd
from cytotable import convert, presets
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from parsl.config import Config
from parsl.executors import HighThroughputExecutor

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    args = parse_args()
    well_fov = args["well_fov"]
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T2"
    well_fov = "C4-1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
input_sqlite_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/{well_fov}.duckdb"
).resolve(strict=True)
destination_sc_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/sc_profiles_{well_fov}.parquet"
).resolve()
destination_organoid_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/organoid_profiles_{well_fov}.parquet"
).resolve()
destination_nucleocentric_parquet_file = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/0.converted_profiles/{well_fov}/nucleocentric_profiles_{well_fov}.parquet"
).resolve()
destination_sc_parquet_file.parent.mkdir(parents=True, exist_ok=True)
dest_datatype = "parquet"

In [4]:
# show the tables
with duckdb.connect(input_sqlite_file) as con:
    tables = con.execute("SHOW TABLES").fetchdf()
    print(tables)
    nuclei_table = con.sql("SELECT * FROM Nuclei").df()
    cells_table = con.sql("SELECT * FROM Cell").df()
    cytoplasm_table = con.sql("SELECT * FROM Cytoplasm").df()
    organoid_table = con.sql("SELECT * FROM Organoid").df()
    nucleocentric_table = con.sql("SELECT * FROM Nucleocentric").df()

            name
0           Cell
1      Cytoplasm
2         Nuclei
3  Nucleocentric
4       Organoid


In [5]:
nuclei_id_set = set(nuclei_table["object_id"].to_list())
cells_id_set = set(cells_table["object_id"].to_list())
cytoplasm_id_set = set(cytoplasm_table["object_id"].to_list())
# find the intersection of the three sets
intersection_set = nuclei_id_set.intersection(cells_id_set, cytoplasm_id_set)
# keep only the rows in the three tables that are in the intersection set
nuclei_table = nuclei_table[nuclei_table["object_id"].isin(intersection_set)]
cells_table = cells_table[cells_table["object_id"].isin(intersection_set)]
cytoplasm_table = cytoplasm_table[cytoplasm_table["object_id"].isin(intersection_set)]

In [6]:
# connect to DuckDB and register the tables
with duckdb.connect() as con:
    con.register("nuclei", nuclei_table)
    con.register("cells", cells_table)
    con.register("cytoplasm", cytoplasm_table)
    # Merge them with SQL
    merged_df = con.execute("""
        SELECT *
        FROM nuclei
        LEFT JOIN cells USING (object_id)
        LEFT JOIN cytoplasm USING (object_id)
    """).df()

## Reorder object IDs

In [7]:
# replace the object_id with a new unique ID
organoid_table["object_id"] = [i for i in range(1, organoid_table.shape[0] + 1)]
merged_df["object_id"] = [i for i in range(1, merged_df.shape[0] + 1)]
nucleocentric_table["object_id"] = [
    i for i in range(1, nucleocentric_table.shape[0] + 1)
]

In [8]:
# save the organoid data as parquet
print(f"Final organoid data shape: {organoid_table.shape}")
organoid_table.to_parquet(destination_organoid_parquet_file, index=False)
organoid_table.head()

Final organoid data shape: (1, 3961)


,object_id,image_set,Organoid_NoChannel_AreaSizeShape_Volume,Organoid_NoChannel_AreaSizeShape_CenterX,Organoid_NoChannel_AreaSizeShape_CenterY,Organoid_NoChannel_AreaSizeShape_CenterZ,Organoid_NoChannel_AreaSizeShape_BboxVolume,Organoid_NoChannel_AreaSizeShape_MinX,Organoid_NoChannel_AreaSizeShape_MaxX,Organoid_NoChannel_AreaSizeShape_MinY,...,Organoid_Mito_Texture_Variance-3-03-256,Organoid_Mito_Texture_Variance-3-04-256,Organoid_Mito_Texture_Variance-3-05-256,Organoid_Mito_Texture_Variance-3-06-256,Organoid_Mito_Texture_Variance-3-07-256,Organoid_Mito_Texture_Variance-3-08-256,Organoid_Mito_Texture_Variance-3-09-256,Organoid_Mito_Texture_Variance-3-10-256,Organoid_Mito_Texture_Variance-3-11-256,Organoid_Mito_Texture_Variance-3-12-256
0,1,C4-1,1681330.0,900.35209,825.153379,18.670183,3842336.0,530,1198,411,...,37.00767,38.304031,36.981798,38.364917,36.994834,37.002754,36.99441,38.327866,37.003461,37.018354


In [9]:
print(f"Final merged single cell dataframe shape: {merged_df.shape}")
# save the sc data as parquet
merged_df.to_parquet(destination_sc_parquet_file, index=False)
merged_df.head()

Final merged single cell dataframe shape: (5, 11883)


,object_id,image_set,Nuclei_NoChannel_AreaSizeShape_Volume,Nuclei_NoChannel_AreaSizeShape_CenterX,Nuclei_NoChannel_AreaSizeShape_CenterY,Nuclei_NoChannel_AreaSizeShape_CenterZ,Nuclei_NoChannel_AreaSizeShape_BboxVolume,Nuclei_NoChannel_AreaSizeShape_MinX,Nuclei_NoChannel_AreaSizeShape_MaxX,Nuclei_NoChannel_AreaSizeShape_MinY,...,Cytoplasm_ER_Texture_Variance-3-03-256,Cytoplasm_ER_Texture_Variance-3-04-256,Cytoplasm_ER_Texture_Variance-3-05-256,Cytoplasm_ER_Texture_Variance-3-06-256,Cytoplasm_ER_Texture_Variance-3-07-256,Cytoplasm_ER_Texture_Variance-3-08-256,Cytoplasm_ER_Texture_Variance-3-09-256,Cytoplasm_ER_Texture_Variance-3-10-256,Cytoplasm_ER_Texture_Variance-3-11-256,Cytoplasm_ER_Texture_Variance-3-12-256
0,1,C4-1,9928.0,933.750000,982.817486,2.984388,15300.0,888,973,954,...,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309
1,2,C4-1,35716.0,950.018927,647.245688,7.775843,55552.0,918,980,583,...,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309
2,3,C4-1,47013.0,940.153957,731.214281,10.973539,71253.0,896,987,689,...,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309
3,4,C4-1,11702.0,734.449752,1094.235088,8.497693,14884.0,704,765,1064,...,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309,1.390692e-309
4,5,C4-1,10937.0,910.905459,972.491085,9.813751,20935.0,877,956,948,...,1.003386e+03,1.004704e+03,1.040708e+03,9.948174e+02,1.048861e+03,1.044943e+03,9.976161e+02,1.029148e+03,1.044922e+03,9.435667e+02


In [10]:
print(f"Final nucleocentric dataframe shape: {nucleocentric_table.shape}")
# save the nucleocentric data as parquet
nucleocentric_table.to_parquet(destination_nucleocentric_parquet_file, index=False)
nucleocentric_table.head()

Final nucleocentric dataframe shape: (5, 3074)


,object_id,image_set,Nucleocentric_ER_CHAMMI75_Feature0,Nucleocentric_ER_CHAMMI75_Feature1,Nucleocentric_ER_CHAMMI75_Feature10,Nucleocentric_ER_CHAMMI75_Feature100,Nucleocentric_ER_CHAMMI75_Feature101,Nucleocentric_ER_CHAMMI75_Feature102,Nucleocentric_ER_CHAMMI75_Feature103,Nucleocentric_ER_CHAMMI75_Feature104,...,Nucleocentric_DNA_SAMMed3D_Feature90,Nucleocentric_DNA_SAMMed3D_Feature91,Nucleocentric_DNA_SAMMed3D_Feature92,Nucleocentric_DNA_SAMMed3D_Feature93,Nucleocentric_DNA_SAMMed3D_Feature94,Nucleocentric_DNA_SAMMed3D_Feature95,Nucleocentric_DNA_SAMMed3D_Feature96,Nucleocentric_DNA_SAMMed3D_Feature97,Nucleocentric_DNA_SAMMed3D_Feature98,Nucleocentric_DNA_SAMMed3D_Feature99
0,1,C4-1,0.019156,-1.739401,5.004485,3.525002,0.299350,-2.344692,-0.689563,-1.384453,...,-0.005400,-0.110464,-0.007396,-0.011059,0.026875,0.038337,0.039051,0.186796,0.375236,0.179741
1,2,C4-1,-1.571196,-1.820811,6.003607,0.422369,-0.261541,-1.669621,-3.174213,1.143755,...,-0.007149,-0.062501,0.028736,-0.010621,0.035144,0.045945,-0.032070,0.195347,0.349686,0.218726
2,3,C4-1,0.360624,-1.821852,6.035419,-1.381854,0.255795,-1.265000,-2.492608,0.377718,...,-0.006303,-0.102697,0.004119,-0.010507,0.050455,0.035108,-0.016333,0.166130,0.294741,0.226676
3,4,C4-1,-2.082378,-0.670269,4.891862,-0.134080,0.193988,-4.315945,-3.395536,0.116719,...,-0.005942,-0.075369,0.116822,-0.010753,0.030144,-0.000758,-0.016409,0.300854,0.320932,0.181254
4,5,C4-1,2.364192,1.129687,4.910215,0.657670,2.177308,-0.348172,-0.376150,-1.249127,...,-0.005393,-0.054183,0.086169,-0.010869,0.036876,0.039546,0.014089,0.255761,0.325735,0.274638
